# 6 — Batch Predictions from a File

Reads a file of **unlabelled** records, cleans the descriptions with the
recipe the model was trained on, and writes back predicted categories with
raw confidence at every level.

The input needs a description column and nothing else — the category
columns are what this produces. Predicted columns are named
`Predicted Level 1 ID`, `Predicted Level 1 Name` and so on, so they sit
alongside any existing labels rather than colliding with them.

Nothing is refused here and no depth is chosen: every record gets a
prediction and the numbers needed to decide what to do with it. Notebook 5
is the other shape — thresholds applied, uncertain records handed back.

All confidence figures are **raw and uncalibrated**.

## Setup

In [0]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import json
import numpy as np
import pandas as pd

from product_classifier import PipelineConfig, data as data_mod
from product_classifier.artifacts import load_bundle, load_cleaning_config
from product_classifier.cleaning import build_cleaner
from product_classifier.scoring import predict_all_depths, score_descriptions

---
## 1. The model

In [0]:
MODEL_DIR = "../outputs/model_v1"

clf = load_bundle(MODEL_DIR)
ID_LEVELS = clf.level_columns
NAME_LEVELS = [c.replace(" ID", " Name") for c in ID_LEVELS]

print("levels:", ID_LEVELS)
print(f"vocabulary: {len(clf.vectorizer.vocabulary_):,} features")
print(f"paths seen in training: {len(clf.valid_paths_):,}")

levels: ['Level 1 ID', 'Level 2 ID', 'Level 3 ID', 'Level 4 ID']
vocabulary: 60,000 features
paths seen in training: 2,275


In [0]:
# The cleaning recipe travels with the model, so the incoming text is
# cleaned exactly as the training text was. A mismatch here is silent:
# it shows up as a raised out-of-vocabulary rate, not an error.
cleaner = load_cleaning_config(MODEL_DIR)

if cleaner is None:
    print("no cleaning recipe in the bundle — rebuilding locally.")
    print("Re-save from notebook 4 with cleaning_cfg=cleaner to be certain.")
    cleaner = build_cleaner()

LOOKUP_PATH = f"{MODEL_DIR}/hierarchy_lookup.csv"
lookup = pd.read_csv(LOOKUP_PATH, dtype=str) if os.path.exists(LOOKUP_PATH) else None

print(cleaner.describe() if hasattr(cleaner, "describe") else cleaner)
print("lookup:", "loaded" if lookup is not None else "not found — IDs only")

DescriptionCleaner(lists=['color_abbrevs', 'colors', 'sizes', 'unit_abbrevs', 'unit_plurals', 'units'], vocabulary=234,377 words, strip_leading_code=True, uppercase=False)
lookup: loaded


---
## 2. The input file

Only `DESC_COL` is required. `ID_COL` is carried through untouched so the
output can be joined back to the source; set it to `None` if the file has
no identifier, and list anything else you want carried in `KEEP_COLS`.

If the file happens to *already* carry category columns — stale labels you
are re-predicting, say — they pass through untouched and sit beside the
new `Predicted ...` columns for comparison.

In [0]:
!pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
INPUT_PATH = "../data/to_predict.txt"
FILE_FORMAT = "csv"          # "csv" | "excel"
DELIMITER = "|"

DESC_COL = "FullDesc"        # the description to predict from
ID_COL = "Vallen ID"         # carried through; None if there is no id
KEEP_COLS = []               # any other columns to carry through

_cfg = PipelineConfig()
_cfg.data.path = INPUT_PATH
_cfg.data.file_format = FILE_FORMAT
_cfg.data.delimiter = DELIMITER
_cfg.data.text_col = DESC_COL
_cfg.data.id_col = ID_COL

# require_labels=False: the hierarchy columns are what this notebook
# creates, so demanding them would refuse the very input it exists for.
records = data_mod.load_data(_cfg.data, require_labels=False)

print(f"{len(records):,} rows, {records.shape[1]} columns")
print(list(records.columns))
records.head()

145,000 rows, 8 columns
['<ID>', 'ERP Cono', 'Vallen ID', 'ERP ShortDescription1', 'ERP ShortDescription2', 'Level 4 ID', 'Level 4 PPH Name', 'FullDesc']


,<ID>,ERP Cono,Vallen ID,ERP ShortDescription1,ERP ShortDescription2,Level 4 ID,Level 4 PPH Name,FullDesc
0,IF3JN1O5ZG2J,1,781568M3545,M3545 LEESON C6T34FB1C,CAT1101 3450/2850RPM 1HP,L4-12785646,AC Motors,M3545 LEESON C6T34FB1C CAT1101 3450/2850RPM 1HP
1,S-8677296,1,16507834,28997938 MOTOR ELECTRIC,AC 480VAC 3000 RPM SERVO,L4-12785646,AC Motors,28997938 MOTOR ELECTRIC AC 480VAC 3000 RPM SERVO
2,S-10643186,1,10882710,1MB21212AA114AG3 MOTOR,7.5HP 3520 RPM 213T TFEC,L4-12785646,AC Motors,1MB21212AA114AG3 MOTOR 7.5HP 3520 RPM 213T TFEC
3,S-9022772,1,66244100PPG000007202A,MOTOR ELECTRIC AC,3HP 460V 1140 RPM,L4-12785646,AC Motors,MOTOR ELECTRIC AC 3HP 460V 1140 RPM
4,CA_VAL_6776389,55,BAL EM3774T,MOTOR 3PH 10HP 1760RPM,TEFC 575V,L4-12785646,AC Motors,MOTOR 3PH 10HP 1760RPM TEFC 575V


In [0]:
# Fail here rather than three cells later.
missing = [c for c in ([ID_COL] if ID_COL else []) + [DESC_COL] + KEEP_COLS
           if c and c not in records.columns]
if missing:
    raise KeyError(f"{missing} not in the input file. Columns present: "
                   f"{list(records.columns)}")

_blank = records[DESC_COL].isna() | (records[DESC_COL].astype(str).str.strip() == "")
print(f"{int(_blank.sum()):,} rows have no description "
      f"({_blank.mean():.2%}) — they will be predicted from nothing and "
      "flagged unreadable")

_existing = [c for c in ID_LEVELS + NAME_LEVELS if c in records.columns]
if _existing:
    print(f"\ninput already carries {_existing}; these pass through "
          "unchanged, beside the new 'Predicted ...' columns")

0 rows have no description (0.00%) — they will be predicted from nothing and flagged unreadable

input already carries ['Level 4 ID']; these pass through unchanged, beside the new 'Predicted ...' columns


---
## 3. Predict

One call: clean, measure readability, then score at every depth.

In [0]:
PREDICTED_PREFIX = "Predicted "      # "" to name them bare

predictions = predict_all_depths(
    clf, records[DESC_COL],
    cleaning_cfg=cleaner,
    lookup=lookup,
    name_columns=NAME_LEVELS if lookup is not None else None,
    prefix=PREDICTED_PREFIX,
)

# Carry the source columns back through.
_front = pd.DataFrame(index=predictions.index)
if ID_COL:
    _front[ID_COL] = records[ID_COL].to_numpy()
for _c in KEEP_COLS + [c for c in ID_LEVELS + NAME_LEVELS if c in records.columns]:
    _front[_c] = records[_c].to_numpy()

predictions = pd.concat([_front, predictions], axis=1)
print(f"{len(predictions):,} predictions")
predictions.head(15)

145,000 predictions


,Vallen ID,Level 4 ID,raw_text,cleaned_text,n_tokens,n_known,oov_rate,readable,Predicted Level 1 ID,Predicted Level 2 ID,Predicted Level 3 ID,Predicted Level 4 ID,L1 Confidence,L1 Agrees,L2 Confidence,L2 Agrees,L3 Confidence,L3 Agrees,L4 Confidence,Predicted Level 1 Name,Predicted Level 2 Name,Predicted Level 3 Name,Predicted Level 4 Name
0,781568M3545,L4-12785646,M3545 LEESON C6T34FB1C CAT1101 3450/2850RPM 1HP,LEESON C6T34FB1C CAT1101 RPM,7,1,0.857143,True,L1-12810533,L2-12810536,L3-12810573,L4-12810800,0.005985,True,0.005045,True,0.004719,True,0.002704,Tools & Equipment,Power Tools,Grinding Tools & Accessories,Die Grinders - Corded
1,16507834,L4-12785646,28997938 MOTOR ELECTRIC AC 480VAC 3000 RPM SERVO,MOTOR ELECTRIC AC 480VAC RPM SERVO,11,8,0.272727,True,L1-12785504,L2-12785509,L3-12785530,L4-12785646,0.999766,True,0.999766,True,0.999766,True,0.999013,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors
2,10882710,L4-12785646,1MB21212AA114AG3 MOTOR 7.5HP 3520 RPM 213T TFEC,MOTOR RPM TFEC,5,3,0.400000,True,L1-12785504,L2-12785509,L3-12785530,L4-12785646,0.993715,True,0.993715,True,0.993715,True,0.930538,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors
3,66244100PPG000007202A,L4-12785646,MOTOR ELECTRIC AC 3HP 460V 1140 RPM,MOTOR ELECTRIC AC 460V RPM,9,7,0.222222,True,L1-12785504,L2-12785509,L3-12785530,L4-12785646,0.999602,True,0.999602,True,0.999602,True,0.998489,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors
4,BAL EM3774T,L4-12785646,MOTOR 3PH 10HP 1760RPM TEFC 575V,MOTOR 3PH 1760RPM TEFC 575V,9,9,0.000000,True,L1-12785504,L2-12785509,L3-12785530,L4-12785646,1.000000,True,1.000000,True,1.000000,True,1.000000,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors
5,BAL M3615T,L4-12785646,MOTOR 3PH 5HP 1800RPM TEFC 208-230/460V,MOTOR 3PH 1800RPM TEFC V,7,5,0.285714,True,L1-12785504,L2-12785509,L3-12785530,L4-12785646,0.999970,True,0.999970,True,0.999970,True,0.999969,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors
6,781568VEBNM3538,L4-12785646,VEBNM3538 MOTOR .5HP 3PH 1765RPM 60HZ 56C 3516M,MOTOR 3PH 1765RPM 60HZ 56C,9,6,0.333333,True,L1-12785504,L2-12785509,L3-12785530,L4-12785646,0.999973,True,0.999973,True,0.999973,True,0.999970,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors
7,16615825,L4-12785646,SGM-08AWFJ91 MOTOR SERVO,MOTOR SERVO,3,3,0.000000,True,L1-12785504,L2-12785509,L3-12785530,L4-12785650,0.925017,True,0.925017,True,0.925017,True,0.529563,Bearings & Power Transmission,Motors & Motor Components,Motors,Special-Purpose Motors
8,BAL CEM4104T,L4-12785646,MOTOR 3PH 30HP 1760RPM TEFC 230/460V,MOTOR 3PH 1760RPM TEFC V,7,7,0.000000,True,L1-12785504,L2-12785509,L3-12785530,L4-12785646,0.999999,True,0.999999,True,0.999999,True,0.999999,Bearings & Power Transmission,Motors & Motor Components,Motors,AC Motors
9,10547956,L4-12785646,R88M-G3K030T-BS2 MOTOR SERVO,MOTOR SERVO,3,3,0.000000,True,L1-12785504,L2-12785509,L3-12785530,L4-12785650,0.925017,True,0.925017,True,0.925017,True,0.529563,Bearings & Power Transmission,Motors & Motor Components,Motors,Special-Purpose Motors


### Reading the confidence columns

`L1 Confidence` … `L4 Confidence` are the raw probabilities of the best
prefix at each depth.

**They are not comparable between levels.** A depth-1 figure and a
depth-4 figure come from different distributions, so a smaller number at
one depth is not a lower bar than a larger number at another. Compare a
level against its own threshold, never against another level.

`L1 Agrees` … `L3 Agrees` flag where the best prefix at that depth is
*not* the first *d* levels of the full-depth answer. Summing over a
prefix's children can let a different prefix win, so those are the
products where truncating would change the answer rather than just
shorten it.

In [0]:
_conf = [f"L{d} Confidence" for d in range(1, len(ID_LEVELS) + 1)]
predictions[_conf].describe().T

,count,mean,std,min,25%,50%,75%,max
L1 Confidence,145000.0,0.766963,0.339188,0.000007,0.581797,0.984824,0.999834,1.0
L2 Confidence,145000.0,0.766283,0.340346,0.000007,0.581568,0.984824,0.999834,1.0
L3 Confidence,145000.0,0.761137,0.345885,0.000007,0.555142,0.984803,0.999834,1.0
L4 Confidence,145000.0,0.717568,0.346548,0.000007,0.480764,0.911182,0.998247,1.0


In [0]:
_agree = [c for c in predictions.columns if c.endswith(" Agrees")]
if _agree:
    print("share where the depth's own best prefix matches the full path:")
    print(predictions[_agree].mean().to_string())

share where the depth's own best prefix matches the full path:
L1 Agrees    0.988952
L2 Agrees    0.987959
L3 Agrees    0.986897


### Records the model could not read

Every word unknown to the model. These still carry a prediction and a
confidence figure, both drawn from how common each category is rather than
from the product — so they should not be accepted whatever the number
says.

In [0]:
unreadable = predictions[~predictions["readable"]]
print(f"{len(unreadable):,} of {len(predictions):,} "
      f"({len(unreadable) / len(predictions):.2%}) have no recognisable word")
print(f"mean out-of-vocabulary rate overall: {predictions['oov_rate'].mean():.2%}")
unreadable.head(15)

1,295 of 145,000 (0.89%) have no recognisable word
mean out-of-vocabulary rate overall: 19.15%


,Vallen ID,Level 4 ID,raw_text,cleaned_text,n_tokens,n_known,oov_rate,readable,Predicted Level 1 ID,Predicted Level 2 ID,Predicted Level 3 ID,Predicted Level 4 ID,L1 Confidence,L1 Agrees,L2 Confidence,L2 Agrees,L3 Confidence,L3 Agrees,L4 Confidence,Predicted Level 1 Name,Predicted Level 2 Name,Predicted Level 3 Name,Predicted Level 4 Name
28,781568ECP3661T-4,L4-12785646,MTR ECP3661T-4,ECP3661T4,1,0,1.0,False,L1-12783695,L2-12783697,L3-12783716,L4-12784313,0.000148,False,0.000148,False,0.000103,False,0.000033,Cutting Tools,Holemaking,General Purpose Drill Bits,Jobber Power Drill Bits
166,NTN 66261138138,L4-13482894,(DC) FLEXLOK 2IN 180G,FLEXLOK,1,0,1.0,False,L1-12783695,L2-12783697,L3-12783716,L4-12784313,0.000148,False,0.000148,False,0.000103,False,0.000033,Cutting Tools,Holemaking,General Purpose Drill Bits,Jobber Power Drill Bits
844,78500723735,L4-12810945,PS5269-XGRY,,0,0,0.0,False,L1-12783695,L2-12783697,L3-12783716,L4-12784313,0.000148,False,0.000148,False,0.000103,False,0.000033,Cutting Tools,Holemaking,General Purpose Drill Bits,Jobber Power Drill Bits
917,669635010817,L4-12783234,"BANDAGES,WOVEN,ASSORTMEN T 20/BOX 01-5050",T,0,0,0.0,False,L1-12783695,L2-12783697,L3-12783716,L4-12784313,0.000148,False,0.000148,False,0.000103,False,0.000033,Cutting Tools,Holemaking,General Purpose Drill Bits,Jobber Power Drill Bits
943,NOR UB144516,L4-12783234,(DV)BANDAID 3/4IN X 3IN,IN,0,0,0.0,False,L1-12783695,L2-12783697,L3-12783716,L4-12784313,0.000148,False,0.000148,False,0.000103,False,0.000033,Cutting Tools,Holemaking,General Purpose Drill Bits,Jobber Power Drill Bits
1455,015451CA155263,L4-13525386,CA155263 SWITCH-REG,SWITCHREG,1,0,1.0,False,L1-12783695,L2-12783697,L3-12783716,L4-12784313,0.000148,False,0.000148,False,0.000103,False,0.000033,Cutting Tools,Holemaking,General Purpose Drill Bits,Jobber Power Drill Bits
1750,662805CL-27-LP,L4-12810712,L-PIN,,0,0,0.0,False,L1-12783695,L2-12783697,L3-12783716,L4-12784313,0.000148,False,0.000148,False,0.000103,False,0.000033,Cutting Tools,Holemaking,General Purpose Drill Bits,Jobber Power Drill Bits
2331,66235593973,L4-13338191,132 AP-80 7/16X1/2 1ALUMINUM-P 93973,AP80 1ALUMINUMP,3,0,1.0,False,L1-12783695,L2-12783697,L3-12783716,L4-12784313,0.000148,False,0.000148,False,0.000103,False,0.000033,Cutting Tools,Holemaking,General Purpose Drill Bits,Jobber Power Drill Bits
2333,66235593952,L4-13338191,132 AP-32 3/8X3/8 1ALUMINUM-PT 93952,AP32 1ALUMINUMPT,3,0,1.0,False,L1-12783695,L2-12783697,L3-12783716,L4-12784313,0.000148,False,0.000148,False,0.000103,False,0.000033,Cutting Tools,Holemaking,General Purpose Drill Bits,Jobber Power Drill Bits
2366,66235593953,L4-13338191,132 AP-33 7/16X3/8 1ALUMINUM-P 93953,AP33 1ALUMINUMP,3,0,1.0,False,L1-12783695,L2-12783697,L3-12783716,L4-12784313,0.000148,False,0.000148,False,0.000103,False,0.000033,Cutting Tools,Holemaking,General Purpose Drill Bits,Jobber Power Drill Bits


---
## 4. Optional: apply thresholds

If notebook 3 wrote a threshold record, the same records can also be run
through the cascade — deepest level clearing its bar, truncated below
that, refused where nothing clears.

In [0]:
THRESHOLDS_PATH = "../outputs/thresholds.json"

if os.path.exists(THRESHOLDS_PATH):
    with open(THRESHOLDS_PATH, encoding="utf-8") as fh:
        _rec = json.load(fh)
    THRESHOLDS = {int(k): float(v) for k, v in _rec["thresholds"].items()}
    print(f"loaded {_rec.get('basis', '?')}-based thresholds "
          f"({_rec.get('scale', 'raw')}):")
    for _d, _t in sorted(THRESHOLDS.items()):
        print(f"  {ID_LEVELS[_d - 1]:<16} {_t:.6f}")
else:
    THRESHOLDS = None
    print(f"no {THRESHOLDS_PATH} — run notebook 3 to produce one.")

no ../outputs/thresholds.json — run notebook 3 to produce one.


In [0]:
if THRESHOLDS:
    cascaded = score_descriptions(
        clf, records[DESC_COL],
        thresholds=THRESHOLDS,
        cleaning_cfg=cleaner,
        min_known_tokens=1,
        lookup=lookup,
        name_columns=NAME_LEVELS if lookup is not None else None,
    )
    if ID_COL:
        cascaded.insert(0, ID_COL, records[ID_COL].to_numpy())

    display(cascaded["status"].value_counts().rename("n_records"))
    display(cascaded.head(15))

---
## 5. Save

In [0]:
OUTPUT_PATH = "../outputs/predictions.csv"

predictions.to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(predictions):,} rows to {OUTPUT_PATH}")
print(f"columns: {list(predictions.columns)}")

wrote 145,000 rows to ../outputs/predictions.csv
columns: ['Vallen ID', 'Level 4 ID', 'raw_text', 'cleaned_text', 'n_tokens', 'n_known', 'oov_rate', 'readable', 'Predicted Level 1 ID', 'Predicted Level 2 ID', 'Predicted Level 3 ID', 'Predicted Level 4 ID', 'L1 Confidence', 'L1 Agrees', 'L2 Confidence', 'L2 Agrees', 'L3 Confidence', 'L3 Agrees', 'L4 Confidence', 'Predicted Level 1 Name', 'Predicted Level 2 Name', 'Predicted Level 3 Name', 'Predicted Level 4 Name']


In [0]:
if THRESHOLDS:
    CASCADE_PATH = "../outputs/predictions_thresholded.csv"
    cascaded.to_csv(CASCADE_PATH, index=False)
    print(f"wrote {len(cascaded):,} rows to {CASCADE_PATH}")